# Standard Logistic Regression - Gallstone Dataset\n\nTrains a standard (non-private) Logistic Regression model on the preprocessed Gallstone dataset.


## 1. Import Libraries


In [1]:
import time
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, f1_score, precision_score, recall_score
import pickle
import json
import os
import warnings
warnings.filterwarnings('ignore')

# Shared metric collector (repository root): logs training accuracy,
# balanced accuracy and AUROC alongside the metrics already tracked below.
import os as _os
import sys as _sys
_REPO_ROOT = _os.path.abspath(_os.path.join('..', '..'))
if _REPO_ROOT not in _sys.path:
    _sys.path.insert(0, _REPO_ROOT)
from metrics_utils import ExtraMetrics


## 2. Load Preprocessed Data


In [2]:
with open('../data/processed_data.pkl', 'rb') as f:
    data = pickle.load(f)

X_train = data['X_train']
X_test = data['X_test']
y_train = data['y_train']
y_test = data['y_test']

# DP norm precondition: clip every row to unit L2 norm (||x_i||_2 <= 1).
X_train = X_train / np.maximum(np.linalg.norm(X_train, axis=1, keepdims=True), 1.0)
X_test = X_test / np.maximum(np.linalg.norm(X_test, axis=1, keepdims=True), 1.0)

print(f"Train features: {X_train.shape}")
print(f"Test features: {X_test.shape}")

Train features: (255, 38)
Test features: (64, 38)


## 3. Train Standard Logistic Regression


In [3]:
lr = LogisticRegression(max_iter=1000, random_state=42)
_t0 = time.perf_counter()
lr.fit(X_train, y_train)
train_time = time.perf_counter() - _t0
# Score the non-private fit on the training partition too, so the
# generalisation gap can be read next to the held-out metrics.
y_train_pred = lr.predict(X_train)
y_pred = lr.predict(X_test)

extra = ExtraMetrics()
accuracy = accuracy_score(y_test, y_pred)
extra.add(y_test, y_pred, model=lr, X=X_test, train_true=y_train, train_pred=y_train_pred, train_time=train_time)
f1 = f1_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)

print(f"Accuracy:  {accuracy:.4f}")
print(f"F1-Score:  {f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")


Accuracy:  0.7656
F1-Score:  0.7619
Precision: 0.7742
Recall:    0.7500


## 4. Save Baseline Results


In [4]:
os.makedirs('output', exist_ok=True)

baseline_results = {
    'model': 'Standard Logistic Regression',
    'dataset': 'Gallstone',
    'accuracy': accuracy,
    'f1_score': f1,
    'precision': precision,
    'recall': recall
}
baseline_results.update(extra.means())

pd.DataFrame([baseline_results]).to_csv('output/baseline_results.csv', index=False)
with open('output/std_lr_report.json', 'w') as f:
    json.dump(baseline_results, f, indent=4)

print("Baseline saved.")

# Export model for MIA
import sys
import os
sys.path.append(os.path.abspath(os.path.join('..', '..')))
from model_exporter import export_model
export_model(lr, 'output/model/std_lr_model.pkl')


print("Added metrics:")
print(extra.describe())

Baseline saved.


Model successfully exported to output/model/std_lr_model.pkl
Added metrics:
  Train Acc:      0.7412 ± 0.0000
  Balanced Acc:   0.7656 ± 0.0000
  AUROC:          0.8359 ± 0.0000
  Train Time (s): 0.0024 ± 0.0000
